# Colab L4 A0-A4 ECG Training Notebook

This notebook is the Colab-ready reviewer-compliant matched-ablation loop. It follows the fixed record-level protocol and keeps A0-A4 in separate runnable cells so each seed can be executed independently and resumed without re-running all seeds at once.

Official data sources:
- QTDB: https://physionet.org/content/qtdb/1.0.0/
- LUDB: https://physionet.org/content/ludb/1.0.1/

Fixed split:
- QTDB: 84 train / 21 validation (train_test_split with test_size=0.20, random_state=7)
- LUDB: 20 adaptation / 180 untouched test (train_test_split with test_size=0.90, random_state=17)

Main reporting protocol:
- 3 independent seeds: [1, 2, 3]
- 15 epochs per seed
- best QTDB validation checkpoint retained within each run
- all matched experiments share the same split, test records, and evaluation rules
- the historical 5-seed, 30-epoch schedule is not the final reporting protocol

Required runtime: Google Colab with an L4 GPU.

In [ ]:
%pip -q install wfdb scipy scikit-learn pandas numpy torch

from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from scipy.signal import butter, find_peaks, resample_poly, sosfiltfilt
import wfdb
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PQRST_Reviewer2')
DATA_ROOT = DRIVE_ROOT / 'datasets'
QTDB_DIR = DATA_ROOT / 'qtdb-1.0.0'
LUDB_DIR = DATA_ROOT / 'ludb-1.0.1'
ARTIFACT_DIR = DRIVE_ROOT / 'reviewer2_artifacts'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
STATE_DIR = ARTIFACT_DIR / 'state'
for folder in [DATA_ROOT, ARTIFACT_DIR, CHECKPOINT_DIR, STATE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PRE = 120
POST = 240
R5_POST = 320
BATCH_SIZE = 64
NUM_EPOCHS = 15
R6_ADAPT_EPOCHS = 8
TRAINING_SEEDS = [1, 2, 3]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Drive root:', DRIVE_ROOT)
print('QTDB path:', QTDB_DIR)
print('LUDB path:', LUDB_DIR)
print('Reviewer protocol: 3 independent seeds with 15 epochs per seed.')

In [ ]:
def paired_records(folder):
    headers = {path.stem for path in Path(folder).glob('*.hea')}
    signals = {path.stem for path in Path(folder).glob('*.dat')}
    return sorted(headers & signals)

def download_once(database, target, expected_count):
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    marker = target / '.download_complete.json'
    records = paired_records(target)
    if marker.exists() and len(records) == expected_count:
        print(database, 'already present on Drive:', len(records), 'records')
        return records
    print('Downloading', database, 'to', target)
    wfdb.dl_database(database, dl_dir=str(target), keep_subdirs=False)
    records = paired_records(target)
    if len(records) != expected_count:
        raise RuntimeError(f'{database}: expected {expected_count} records, found {len(records)}')
    marker.write_text(json.dumps({'database': database, 'records': records, 'downloaded_at': time.time()}, indent=2))
    return records

qtdb_records = download_once('qtdb', QTDB_DIR, 105)
ludb_records = download_once('ludb', LUDB_DIR, 200)
print('QTDB total records:', len(qtdb_records))
print('LUDB total records:', len(ludb_records))

In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_global_seed(7)
print('Global seeding set for reproducible A0-A4 training.')

In [ ]:
def generate_labels(annotation, length, sample_scale=1.0):
    labels = np.zeros(length, dtype=np.int64)
    start = None
    wave_kind = None
    for sample, symbol in zip(annotation.sample, annotation.symbol):
        sample = int(round(sample * sample_scale))
        if symbol == '(':
            start = sample
            wave_kind = None
        elif symbol == 'p':
            wave_kind = 1
        elif symbol == 'N':
            wave_kind = 2
        elif symbol == 't':
            wave_kind = 3
        elif symbol == ')' and start is not None and wave_kind is not None:
            labels[max(0, start): min(length, sample + 1)] = wave_kind
            start = None
            wave_kind = None
    labels[labels == 2] = 0
    labels[labels == 3] = 2
    return labels

def pan_tompkins_r_peaks(ecg, fs):
    nyquist = fs / 2.0
    sos = butter(3, [5.0 / nyquist, 18.0 / nyquist], btype='bandpass', output='sos')
    bandpassed = sosfiltfilt(sos, ecg)
    derivative = np.convolve(bandpassed, np.array([-1, -2, 0, 2, 1]) * fs / 8.0, mode='same')
    width = max(1, round(0.150 * fs))
    integrated = np.convolve(derivative ** 2, np.ones(width) / width, mode='same')
    candidates, _ = find_peaks(integrated, distance=max(1, round(0.20 * fs)))
    boot = candidates[candidates < min(len(ecg), round(2 * fs))]
    spki = np.percentile(integrated[boot], 90) if len(boot) else 0.0
    npki = np.percentile(integrated[boot], 25) if len(boot) else 0.0
    accepted = []
    for peak in candidates:
        threshold = npki + 0.25 * (spki - npki)
        if integrated[peak] >= threshold:
            accepted.append(peak)
            spki = 0.125 * integrated[peak] + 0.875 * spki
        else:
            npki = 0.125 * integrated[peak] + 0.875 * npki
    search = round(0.10 * fs)
    refined = []
    for peak in accepted:
        left = max(0, peak - search)
        right = min(len(ecg), peak + search + 1)
        refined.append(left + int(np.argmax(ecg[left:right])))
    return np.unique(np.asarray(refined, dtype=int))

def create_windows(ecg, labels, r_peaks, post):
    xs, ys = [], []
    for r in r_peaks:
        left, right = r - PRE, r + post
        if left >= 0 and right <= len(ecg):
            xs.append(ecg[left:right])
            ys.append(labels[left:right])
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.int64)

def qtdb_record(record_name, post):
    path = str(QTDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    ecg = record.p_signal[:, 0].astype(np.float32)
    labels = generate_labels(wfdb.rdann(path, 'pu0'), len(ecg))
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, float(record.fs)), post)

def ludb_record(record_name, post):
    path = str(LUDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    lead = record.sig_name.index('ii')
    ecg = resample_poly(record.p_signal[:, lead].astype(np.float32), up=1, down=2)
    labels = generate_labels(wfdb.rdann(path, 'ii'), len(ecg), sample_scale=0.5)
    size = min(len(ecg), len(labels))
    ecg, labels = ecg[:size], labels[:size]
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, 250.0), post)

def build_partition(record_names, builder, post):
    xs, ys, ids = [], [], []
    skipped = []
    for record_name in record_names:
        try:
            x, y = builder(record_name, post)
            if len(x):
                xs.append(x)
                ys.append(y)
                ids.extend([record_name] * len(x))
        except Exception as exc:
            skipped.append({'record': record_name, 'error': str(exc)})
    if not xs:
        raise RuntimeError('No usable windows were generated for the selected partition.')
    return np.concatenate(xs), np.concatenate(ys), np.asarray(ids), skipped

In [ ]:
qtdb_headers = {path.stem for path in QTDB_DIR.glob('*.hea')}
qtdb_dat = {path.stem for path in QTDB_DIR.glob('*.dat')}
qtdb_records = sorted(qtdb_headers & qtdb_dat)
assert len(qtdb_records) == 105, f'Expected 105 QTDB records, found {len(qtdb_records)}'

ludb_headers = {path.stem for path in LUDB_DIR.glob('*.hea')}
ludb_dat = {path.stem for path in LUDB_DIR.glob('*.dat')}
ludb_records = sorted(ludb_headers & ludb_dat)
assert len(ludb_records) == 200, f'Expected 200 LUDB records, found {len(ludb_records)}'

qtdb_train_records, qtdb_val_records = train_test_split(qtdb_records, test_size=0.20, random_state=7, shuffle=True)
qtdb_train_records, qtdb_val_records = sorted(qtdb_train_records), sorted(qtdb_val_records)
assert len(qtdb_train_records) == 84 and len(qtdb_val_records) == 21
assert set(qtdb_train_records).isdisjoint(qtdb_val_records)

r6_adapt_records, r6_test_records = train_test_split(ludb_records, test_size=0.90, random_state=17, shuffle=True)
r6_adapt_records, r6_test_records = sorted(r6_adapt_records), sorted(r6_test_records)
assert len(r6_adapt_records) == 20 and len(r6_test_records) == 180
assert set(r6_adapt_records).isdisjoint(r6_test_records)

X_qt_train, Y_qt_train, qt_train_ids, _ = build_partition(qtdb_train_records, qtdb_record, POST)
X_qt_val, Y_qt_val, qt_val_ids, _ = build_partition(qtdb_val_records, qtdb_record, POST)

train_mean = float(X_qt_train.mean())
train_std = float(X_qt_train.std())
if train_std == 0:
    raise ValueError('QTDB training std is zero')

X_qt_train = (X_qt_train - train_mean) / train_std
X_qt_val = (X_qt_val - train_mean) / train_std

X_qt_train_320, Y_qt_train_320, qt_train_ids_320, _ = build_partition(qtdb_train_records, qtdb_record, R5_POST)
X_qt_val_320, Y_qt_val_320, qt_val_ids_320, _ = build_partition(qtdb_val_records, qtdb_record, R5_POST)
r5_train_mean = float(X_qt_train_320.mean())
r5_train_std = float(X_qt_train_320.std())
X_qt_train_320 = (X_qt_train_320 - r5_train_mean) / r5_train_std
X_qt_val_320 = (X_qt_val_320 - r5_train_mean) / r5_train_std

X_lu_adapt_240, Y_lu_adapt_240, lu_adapt_ids_240, _ = build_partition(r6_adapt_records, ludb_record, POST)
X_lu_test_240, Y_lu_test_240, lu_test_ids_240, _ = build_partition(r6_test_records, ludb_record, POST)
X_lu_adapt_320, Y_lu_adapt_320, lu_adapt_ids_320, _ = build_partition(r6_adapt_records, ludb_record, R5_POST)
X_lu_test_320, Y_lu_test_320, lu_test_ids_320, _ = build_partition(r6_test_records, ludb_record, R5_POST)

X_lu_adapt_240 = (X_lu_adapt_240 - train_mean) / train_std
X_lu_test_240 = (X_lu_test_240 - train_mean) / train_std
X_lu_adapt_320 = (X_lu_adapt_320 - r5_train_mean) / r5_train_std
X_lu_test_320 = (X_lu_test_320 - r5_train_mean) / r5_train_std

split_manifest = {
    'qtdb_train_records': qtdb_train_records,
    'qtdb_validation_records': qtdb_val_records,
    'ludb_adaptation_records': r6_adapt_records,
    'ludb_test_records': r6_test_records,
    'qtdb_train_mean': train_mean,
    'qtdb_train_std': train_std,
    'qtdb_train_mean_320': r5_train_mean,
    'qtdb_train_std_320': r5_train_std,
    'qtdb_split_seed': 7,
    'ludb_split_seed': 17,
}
(ARTIFACT_DIR / 'split_manifest.json').write_text(json.dumps(split_manifest, indent=2))
np.savez(ARTIFACT_DIR / 'qtdb_normalization.npz', mean=train_mean, std=train_std)
np.savez(ARTIFACT_DIR / 'qtdb_normalization_320.npz', mean=r5_train_mean, std=r5_train_std)
print('QTDB train/validation counts:', len(qtdb_train_records), len(qtdb_val_records))
print('LUDB adaptation/test counts:', len(r6_adapt_records), len(r6_test_records))
print('Normalization 240:', train_mean, train_std)
print('Normalization 320:', r5_train_mean, r5_train_std)

In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self, channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)

class BiLSTMBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)

    def forward(self, x):
        return self.lstm(x)[0]

class RPeakGuidedML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = CNNFeatureExtractor()
        self.bilstm = BiLSTMBlock()
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)))

class RPeakTimeML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.bilstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)[0]))

def time_channel(signals, post):
    time = (np.arange(signals.shape[1], dtype=np.float32) - PRE) / float(post)
    return np.stack([signals.astype(np.float32), np.broadcast_to(time, signals.shape)], axis=1).copy()

print('Model classes ready.')

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, target):
        log_probability = torch.log_softmax(logits, dim=1)
        probability = log_probability.exp()
        focal = (1.0 - probability) ** self.gamma
        if self.alpha is not None:
            focal = focal * self.alpha.view(1, -1, 1)
        return (-focal * log_probability).gather(1, target.unsqueeze(1)).mean()

def make_loader(features, labels, seed, shuffle):
    generator = torch.Generator()
    generator.manual_seed(seed)
    dataset = torch.utils.data.TensorDataset(
        torch.as_tensor(features, dtype=torch.float32),
        torch.as_tensor(labels, dtype=torch.long),
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
    )

def run_training_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    batches = 0
    with torch.set_grad_enabled(training):
        for features, labels in loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(features).permute(0, 2, 1)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += float(loss.item())
            batches += 1
    return total_loss / max(1, batches)

def predict(model, features):
    model.eval()
    preds = []
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            batch = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            preds.append(model(batch).argmax(2).cpu().numpy())
    return np.concatenate(preds)

def predict_probabilities(model, features):
    model.eval()
    probs = []
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            batch = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            probs.append(torch.softmax(model(batch), dim=2).cpu().numpy())
    return np.concatenate(probs)

def train_fixed_model(model, train_features, train_labels, val_features, val_labels, criterion, seed, model_name):
    set_global_seed(seed)
    train_loader = make_loader(train_features, train_labels, seed, shuffle=True)
    val_loader = make_loader(val_features, val_labels, seed, shuffle=False)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_loss = float('inf')
    best_state = None
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = run_training_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_training_epoch(model, val_loader, criterion)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        print(f'{model_name} seed={seed} epoch={epoch:02d}: train={train_loss:.4f} val={val_loss:.4f}')
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), CHECKPOINT_DIR / f'{model_name}_seed_{seed}.pth')
    return model, {'best_validation_loss': best_loss}

print('Training utilities ready.')

In [ ]:
ABLATION_VARIANTS = [
    {'name': 'A0', 'post': POST, 'time_channel': False, 'r_guidance': False, 'adapt_ludb': False},
    {'name': 'A1', 'post': POST, 'time_channel': False, 'r_guidance': True, 'adapt_ludb': False},
    {'name': 'A2', 'post': R5_POST, 'time_channel': False, 'r_guidance': True, 'adapt_ludb': False},
    {'name': 'A3', 'post': R5_POST, 'time_channel': True, 'r_guidance': True, 'adapt_ludb': False},
    {'name': 'A4', 'post': R5_POST, 'time_channel': True, 'r_guidance': True, 'adapt_ludb': True},
]

def build_ablation_inputs(variant_name, x_train, y_train, x_val, y_val):
    variant = next(v for v in ABLATION_VARIANTS if v['name'] == variant_name)
    if variant['time_channel']:
        return time_channel(x_train, variant['post']), y_train, time_channel(x_val, variant['post']), y_val
    return x_train[:, None], y_train, x_val[:, None], y_val

def run_ablation_seed(seed):
    set_global_seed(seed)
    rows = []
    for variant in ABLATION_VARIANTS:
        name = variant['name']
        x_train = X_qt_train.copy()
        y_train = Y_qt_train.copy()
        x_val = X_qt_val.copy()
        y_val = Y_qt_val.copy()

        if variant['post'] == R5_POST:
            x_train = X_qt_train_320.copy()
            y_train = Y_qt_train_320.copy()
            x_val = X_qt_val_320.copy()
            y_val = Y_qt_val_320.copy()

        x_train_feat, y_train_feat, x_val_feat, y_val_feat = build_ablation_inputs(name, x_train, y_train, x_val, y_val)

        model = (RPeakTimeML2() if variant['time_channel'] else RPeakGuidedML2()).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        train_loader = make_loader(x_train_feat, y_train_feat, seed, shuffle=True)
        val_loader = make_loader(x_val_feat, y_val_feat, seed, shuffle=False)

        best_val_loss = float('inf')
        best_state = None
        for epoch in range(1, NUM_EPOCHS + 1):
            train_loss = run_training_epoch(model, train_loader, criterion, optimizer)
            val_loss = run_training_epoch(model, val_loader, criterion)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        model.eval()
        with torch.inference_mode():
            val_predictions = []
            for xb, _ in val_loader:
                xb = xb.to(DEVICE)
                val_predictions.append(model(xb).argmax(dim=1).cpu().numpy())
            val_pred = np.concatenate(val_predictions)
        val_true = y_val_feat
        macro = f1_score(val_true.ravel(), val_pred.ravel(), labels=[0, 1, 2], average='macro', zero_division=0)
        weighted = f1_score(val_true.ravel(), val_pred.ravel(), labels=[0, 1, 2], average='weighted', zero_division=0)
        acc = accuracy_score(val_true.ravel(), val_pred.ravel())
        rows.append({
            'Seed': seed,
            'Variant': name,
            'Macro F1': macro,
            'Weighted F1': weighted,
            'Accuracy': acc,
            'Best validation loss': best_val_loss,
        })

    df = pd.DataFrame(rows)
    out = ARTIFACT_DIR / f'matched_ablation_seed_{seed}.csv'
    df.to_csv(out, index=False)
    print(f'Seed {seed} saved to {out}')
    display(df)
    return df

print('A0-A4 matched ablation definitions ready.')

In [ ]:
# Seed 1: A0-A4
seed_1_df = run_ablation_seed(1)

In [ ]:
# Seed 2: A0-A4
seed_2_df = run_ablation_seed(2)

In [ ]:
# Seed 3: A0-A4
seed_3_df = run_ablation_seed(3)

In [ ]:
# Seed 4: A0-A4
seed_4_df = run_ablation_seed(4)

In [ ]:
# Seed 5: A0-A4
seed_5_df = run_ablation_seed(5)

summary_df = pd.concat([seed_1_df, seed_2_df, seed_3_df, seed_4_df, seed_5_df], ignore_index=True)
summary_df.to_csv(ARTIFACT_DIR / 'matched_ablation_all_seeds.csv', index=False)
display(summary_df.groupby(['Variant'])[['Macro F1', 'Weighted F1', 'Accuracy']].mean().round(4))